In [ ]:
import sympy as sp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def newton_raphson_auto(expr_str, x0, tol=1e-6, max_iter=100):
    # 1. Definir la variable simbólica y la función
    x = sp.Symbol('x')
    f_expr = sp.sympify(expr_str)

    # 2. Calcular la derivada automáticamente con SymPy
    df_expr = sp.diff(f_expr, x)

    # Mostrar la derivada calculada en pantalla
    print(f"Función f(x):  {f_expr}")
    print(f"Derivada f'(x): {df_expr}\n")

    # 3. Convertir expresiones simbólicas a funciones numéricas ejecutable por NumPy
    f = sp.lambdify(x, f_expr, 'numpy')
    df = sp.lambdify(x, df_expr, 'numpy')

    # 4. Algoritmo de Newton-Raphson
    iteraciones = []
    x_n = float(x0)

    for n in range(max_iter):
        fx = f(x_n)
        dfx = df(x_n)

        if n == 0:
            error = np.nan
        else:
            error = abs(x_n - x_prev)

        iteraciones.append({
            'n': n,
            'x_n': x_n,
            'f(x_n)': fx,
            'f\'(x_n)': dfx,
            'Error': error
        })

        if n > 0 and error < tol:
            break

        x_prev = x_n
        x_n = x_n - fx / dfx

    return pd.DataFrame(iteraciones), f

# ==================== EJECUCIÓN DEL CÓDIGO ====================

# Solo define la función como texto (puedes usar sin(x), exp(x), x**3 - x - 1, etc.)
expresion = "x**2 - 2"
x0 = 1.0
tolerancia = 1e-6

# Ejecutar método con derivación automática
df_resultados, f_num = newton_raphson_auto(expresion, x0, tol=tolerancia)

# Mostrar la tabla formateada
print("=== TABLA DE ITERACIONES ===")
display(df_resultados.style.format({
    'x_n': '{:.6f}',
    'f(x_n)': '{:.6f}',
    'f\'(x_n)': '{:.6f}',
    'Error': '{:.6f}'
}))

# Gráfica de convergencia
x_vals = np.linspace(df_resultados['x_n'].min() - 0.5, df_resultados['x_n'].max() + 0.5, 400)
plt.figure(figsize=(8, 4))
plt.plot(x_vals, f_num(x_vals), label=f"$f(x) = {expresion}$", color='blue')
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.scatter(df_resultados['x_n'], df_resultados['f(x_n)'], color='red', zorder=5, label='Iteraciones $x_n$')

for i, row in df_resultados.iterrows():
    plt.annotate(f"$x_{int(row['n'])}$", (row['x_n'], row['f(x_n)']), textcoords="offset points", xytext=(0,10), ha='center')

plt.title('Convergencia del Método de Newton-Raphson')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.show()